# ASO → IDT order format

Turn an antisense oligonucleotide — **sequence** + per-position **sugar chemistry** + **phosphorothioate (PS) pattern** — into an IDT-style order string, using the shared `tauso.common.modifications.to_idt_notation`.

- **Chemical (sugar) pattern** — one letter per base: `M` = 2′-MOE · `C` = cEt · `L` = LNA · `d` = DNA
- **PS pattern** — one character per inter-nucleotide linkage (length = *sequence length − 1*): `*` = phosphorothioate · anything else = phosphodiester. Omit it for an all-PS backbone.

> ⚠️ The emitted modification codes are a starting point — confirm them against IDT's current catalogue before ordering (cEt in particular is a placeholder token).

In [1]:
from tauso.common.modifications import to_idt_notation

# convenience: build a gapmer's chemical (sugar) pattern = modified wings + DNA gap
_SUGAR = {"MOE": "M", "2'-MOE": "M", "cEt": "C", "LNA": "L", "DNA": "d"}

def gapmer_pattern(length, modification="MOE", wing5=5, wing3=5):
    """Chemical pattern for a gapmer: `wing5` modified + DNA gap + `wing3` modified.
    `modification` is the wing sugar: 'MOE', 'cEt', or 'LNA'."""
    gap = length - wing5 - wing3
    if gap < 0:
        raise ValueError(f"wings ({wing5}+{wing3}) exceed length {length}")
    code = _SUGAR[modification]
    return code * wing5 + "d" * gap + code * wing3

## One ASO

Edit the fields and run. For a non-gapmer (mixmer), set `chemical_pattern` directly to any `M`/`C`/`L`/`d` string instead of using `gapmer_pattern`.

In [2]:
sequence         = "GCAGTTATATTAGGTTCTCG"                     # bases, 5'->3'
modification     = "MOE"                                      # wing chemistry: MOE / cEt / LNA
chemical_pattern = gapmer_pattern(len(sequence), modification, wing5=5, wing3=5)  # 5-10-5 gapmer
ps_pattern       = "*" * (len(sequence) - 1)                  # full phosphorothioate

idt = to_idt_notation(sequence, chemical_pattern, ps_pattern)

print("sequence         :", sequence)
print("chemical pattern :", chemical_pattern)
print("PS pattern       :", ps_pattern)
print("IDT order string :", idt)

sequence         : GCAGTTATATTAGGTTCTCG
chemical pattern : MMMMMddddddddddMMMMM
PS pattern       : *******************
IDT order string : /52MOErG/*/i2MOErC/*/i2MOErA/*/i2MOErG/*/i2MOErT/*T*A*T*A*T*T*A*G*G*T*/i2MOErT/*/i2MOErC/*/i2MOErT/*/i2MOErC/*/32MOErG/


## A batch of ASOs

One row per ASO. Build gapmers with `gapmer_pattern(...)`, or pass any explicit `M`/`C`/`L`/`d` pattern (e.g. the mixed MOE/cEt wing below). `to_idt_notation` defaults to an all-PS backbone.

In [3]:
import pandas as pd

# (name, sequence, chemical_pattern)
specs = [
    ("MALAT1_top",   "GCAGTTATATTAGGTTCTCG", gapmer_pattern(20, "MOE", 5, 5)),
    ("v2_3",         "GTTATATTAGGTTCTCGTGT", gapmer_pattern(20, "MOE", 5, 5)),
    ("moe_cet_wing", "GCAGTTATATTAGGTTCTCG", "MMMCCddddddddddCCMMM"),  # mixed MOE/cEt wings
]

pd.DataFrame(
    [{"name": n, "sequence": s, "chemical_pattern": p, "idt_order": to_idt_notation(s, p)}
     for n, s, p in specs]
)

,name,sequence,chemical_pattern,idt_order
0,MALAT1_top,GCAGTTATATTAGGTTCTCG,MMMMMddddddddddMMMMM,/52MOErG/*/i2MOErC/*/i2MOErA/*/i2MOErG/*/i2MOE...
1,v2_3,GTTATATTAGGTTCTCGTGT,MMMMMddddddddddMMMMM,/52MOErG/*/i2MOErT/*/i2MOErT/*/i2MOErA/*/i2MOE...
2,moe_cet_wing,GCAGTTATATTAGGTTCTCG,MMMCCddddddddddCCMMM,/52MOErG/*/i2MOErC/*/i2MOErA/*/icEtG/*/icEtT/*...
